# RepoForge + Greenlight — free Colab workbench

**Purpose:** Rebuild and preview the *existing* RepoForge React/Vite website from its public GitHub repository in an optional Google Colab CPU session, check public GitHub repository metadata, and (only with explicit permission) inspect whether selected Greenlight progress files exist in your own Google Drive.

**Not a replacement for your Ubuntu workstation.** Colab runtimes expire and have usage limits; this notebook does not provide always-on hosting, permanent storage, a free GPU guarantee, background training, or access to your private local checkpoints. Use the normal Vercel/GitHub Pages site for an always-available **static** dashboard.

**Privacy / cost:** No API keys required, no paid AI provider calls, no model downloads, no training, no public tunneling, and no automatic Drive mount or file upload. Public repository metadata only. The older Greenlight R9 multi-AI notebook remains a separate experiment; its paid cloud providers must stay disabled unless you independently opt in. Do not paste passwords, GitHub tokens, recovery codes or checkpoint contents into this notebook.

In [ ]:
from pathlib import Path
import datetime as dt
import json, os, shutil, subprocess, sys, time
REPO_URL = "https://github.com/HenricHuntsman/repo-manager-ultimate.git"
REPO_DIR = Path("/content/repoforge-dashboard")
print("Python:", sys.version.split()[0])
print("Colab session:", dt.datetime.now(dt.timezone.utc).isoformat())
print("GPU not needed for RepoForge static build.")

## 1. Clone source without overwriting existing work

This clones the official public repository into a new temporary folder. It intentionally **does not** reset or pull over an existing checkout.

In [ ]:
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"Refusing to overwrite existing non-Git directory: {REPO_DIR}")
    print("Existing checkout preserved:", REPO_DIR)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", commit)
assert (REPO_DIR / "package.json").is_file(), "Expected RepoForge project source"

## Extra tool: RepoForge Doctor (read-only runtime preflight)

Run the homegrown Python script bundled with the repository. It checks installed Node, Python and any GPU reported by nvidia-smi without installing packages, reading environment secrets or starting training. This output is a **diagnostic**, not proof that Greenlight/MCP is running.

In [ ]:
doctor = REPO_DIR / "tools" / "repoforge_doctor.py"
if not doctor.is_file():
    raise RuntimeError("RepoForge Doctor is missing from the selected Git checkout")
subprocess.run([sys.executable, str(doctor), "runtime"], check=True)
print("RepoForge Doctor finished. No files or model checkpoints were modified.")

## 2. Run the same automated tests and static-site build

Vite 7 needs a sufficiently recent Node.js version. Use Node 22.12+ or Node 24; if Colab's current image has an older version, update the runtime's Node installation yourself, then rerun this cell. The notebook refuses to silently run unsupported Node versions.

In [ ]:
node = shutil.which("node")
npm = shutil.which("npm")
if not node or not npm:
    raise RuntimeError("Node.js and npm are missing; install Node 22.12+ or 24, then rerun.")
version = subprocess.check_output([node, "-v"], text=True).strip().lstrip("v")
major, minor = (int(x) for x in version.split(".")[:2])
if not ((major == 22 and minor >= 12) or major >= 24):
    raise RuntimeError(f"Node {version} unsupported for this Vite version. Use Node 22.12+ or 24.")
env = dict(os.environ, ELECTRON_SKIP_BINARY_DOWNLOAD="1", VITE_PUBLIC_HOSTED="true", CI="1")
steps = [
    ["npm", "ci", "--ignore-scripts", "--no-audit", "--no-fund"],
    ["npm", "test"],
    ["npm", "run", "build"],
]
for command in steps:
    print("\nRunning:", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, env=env, check=True)
assert (REPO_DIR / "dist" / "index.html").is_file()
print("\nPASS — RepoForge public-hosted web build exists. This is not proof of any live deployment.")

## 3. Preview RepoForge inside Colab

This temporary preview is available only while your notebook's Colab runtime remains connected. It does not publish a new Vercel deployment. The deployed website at https://repoforge-dashboard.vercel.app is separate.

In [ ]:
from google.colab import output
from urllib.request import urlopen
from urllib.error import URLError
import atexit
preview_log = open("/content/repoforge-vite-preview.log", "a", encoding="utf-8")
preview = subprocess.Popen(
    ["npm", "run", "preview", "--", "--host", "0.0.0.0", "--port", "4173", "--strictPort"],
    cwd=REPO_DIR, stdout=preview_log, stderr=subprocess.STDOUT, env=env,
)
atexit.register(preview.terminate)
for attempt in range(15):
    if preview.poll() is not None:
        raise RuntimeError("Preview exited. Inspect /content/repoforge-vite-preview.log")
    try:
        with urlopen("http://127.0.0.1:4173/", timeout=1) as response:
            if response.status == 200: break
    except URLError:
        time.sleep(1)
else:
    raise RuntimeError("Preview did not become ready")
print("Local preview is running only within this Colab runtime.")
output.serve_kernel_port_as_iframe(4173, height=650)

## 4. Read-only public GitHub project snapshot

GitHub's unauthenticated API can be rate-limited. Failures are recorded as **unavailable**, not healthy. This reports public repository metadata only and does not inspect private code or execute CI.

In [ ]:
from urllib.request import Request
from urllib.error import HTTPError
snapshot = {"checked_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "owner":"HenricHuntsman", "source_commit":commit, "repositories":[], "status":"unavailable"}
try:
    request = Request("https://api.github.com/users/HenricHuntsman/repos?per_page=100&page=1&type=owner&sort=updated",headers={"Accept":"application/vnd.github+json","User-Agent":"RepoForge-Colab-ReadOnly"})
    with urlopen(request, timeout=15) as response:
        records=json.load(response)
    if not isinstance(records, list):
        raise ValueError("Unexpected GitHub API response")
    snapshot["repositories"]=[{"name":r["name"],"updated_at":r.get("updated_at"),"language":r.get("language"),"size_kb":r.get("size"),"archived":bool(r.get("archived",False))} for r in records if isinstance(r,dict) and isinstance(r.get("name"),str)]
    snapshot["status"]="ok"
except (HTTPError, URLError, ValueError, KeyError, TimeoutError) as exc:
    snapshot["error"]=type(exc).__name__
print("Public scan status:", snapshot["status"], "| repositories:",len(snapshot["repositories"]))
print("Limit: first page (up to 100 public repos); private repositories are not scanned.")
print(json.dumps(snapshot["repositories"][:8],indent=2))

## 5. Optional: check Greenlight evidence in **your own** Drive

Set `INSPECT_MY_DRIVE = True` yourself before running if you want an explicit Drive mount prompt. This **only** checks the names and sizes of specific saved progress files. It does not deserialize model checkpoints, run the R9 paid multi-model notebook, scan your whole Drive, upload files, or assume the historic training is running today.

These reports describe the **12 October 2026** R9.1 CPU tests, R9 optional multi-provider work, and Colab starter. A report existing is not proof of a currently active runtime. Older multi-GB native Greenlight weights and the newer 1.08M-parameter starter are different models.

In [ ]:
INSPECT_MY_DRIVE = False  # explicitly change to True to authorize mounting your Drive
greenlight_inventory = {"status":"not_requested","files":[]}
if INSPECT_MY_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    folder=Path("/content/drive/MyDrive/Greenlight_GitHub_Backup_2026-10-12")
    names=[
        "PROGRESS-MULTI-AI-R9-1-2026-10-12.md",
        "README-MULTI-AI-R9.md",
        "Greenlight_R9_Multi_AI_Colab.ipynb",
        "Greenlight_GPU_Report_2026-10-12.json",
    ]
    greenlight_inventory["status"]="directory_found" if folder.is_dir() else "directory_missing"
    if folder.is_dir():
        for name in names:
            p=folder/name
            greenlight_inventory["files"].append({"name":name,"found":p.is_file(),"bytes":p.stat().st_size if p.is_file() else None})
    print(json.dumps(greenlight_inventory,indent=2))
else:
    print("Greenlight Drive inspection not requested; no Drive access attempted.")

## 6. Optional: save a privacy-limited report in Drive

No saving occurs unless you explicitly opt in AND mounted Drive in step 5. The report includes public GitHub metadata and *file-presence results* only. It never contains secrets, notebook output, model weights, answers or private source code.

In [ ]:
SAVE_REPORT_TO_DRIVE = False
if SAVE_REPORT_TO_DRIVE:
    if not INSPECT_MY_DRIVE:
        raise RuntimeError("Mount Drive in step 5 first; saving is opt-in.")
    folder=Path("/content/drive/MyDrive/RepoForge_Colab_Reports")
    folder.mkdir(parents=True,exist_ok=True)
    report={"schema":1,"generated_utc":dt.datetime.now(dt.timezone.utc).isoformat(),"github":snapshot,"greenlight_inventory":greenlight_inventory}
    filename="repoforge_"+dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")+".json"
    target=folder/filename
    with target.open("x",encoding="utf-8") as f:
        json.dump(report,f,indent=2)
    print("Saved read-only report:",target.name)
else:
    print("No report saved. Your Colab session is ephemeral.")

## What next?

- The stable website remains at [RepoForge](https://repoforge-dashboard.vercel.app). The notebook is a **temporary developer workbench**, not the always-on host.
- Run full Greenlight R9/Flask/MCP tests only in its **separate** own-source Colab notebook, using the archived R9.1 version and user-approved dependencies. Do not enable paid cloud provider calls by default.
- Validate the Greenlight native model on held-out tasks before describing it as capable. The preserved step-595 mini-AGI starter scored 0/20 on its brief arithmetic check.
- Keep all original checkpoint and optimizer artifacts immutable; use copies for any later resume experiments.
